# T05 — One Stage III backward Euler-Maruyama step, manual check

Scope (per `tasks.md` T05 and the 2026-09-22 task instructions):

```text
X_new = X_old + (-U + grad(K)) * dt + sqrt(2*K*dt) * xi,   xi ~ N(0, I_2)
log_w_new = log_w_old - div(U) * dt
```

with `U(x,t) = (uo(x,t), vo(x,t))` (T01.5-locked, windage excluded), `grad(K)`/`div(U)`
from T03, T02's interpolation reused for `U` and for evaluating the derivative grids at
a particle's position, and T04's stop-on-invalid boundary policy reused unchanged.

One particle only, one step only -- no multi-step trajectory (T06), no forcing ensemble,
no multi-particle simulation, no 7-day run.

In [9]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np

from iste.environment import GriddedFieldSource
from iste.geometry import metres_per_degree_latitude, metres_per_degree_longitude
from iste.step import WeightedParticleState, backward_euler_maruyama_step

LAT = np.array([10.0, 20.0, 30.0])
LON = np.array([100.0, 110.0, 120.0])
TIME = np.array(["2020-01-01T00:00", "2020-01-01T01:00"], dtype="datetime64[ns]")
T0 = TIME[0]
DT = 900.0  # 15 minutes, seconds
ZERO_UV = np.zeros((3, 3))
ZERO_K = np.zeros((3, 3))


def make_ocean(uo_grid, vo_grid):
    uo = np.stack([uo_grid, uo_grid])
    vo = np.stack([vo_grid, vo_grid])
    return GriddedFieldSource(
        source_path="<synthetic>", time_name="time", time=TIME, lat_name="lat", lat=LAT,
        lon_name="lon", lon=LON, data={"uo": uo, "vo": vo}, fill_threshold=1e19,
    )


def show(label, before, after):
    print(f"{label}:")
    print(f"  before: lat={before.lat[0]:.8f}, lon={before.lon[0]:.8f}, "
          f"log_w={before.log_weight[0]:.6f}, valid={before.valid[0]}")
    print(f"  after:  lat={after.lat[0]:.8f}, lon={after.lon[0]:.8f}, "
          f"log_w={after.log_weight[0]:.6f}, valid={after.valid[0]}")

## 1. Deterministic step, compared against the analytically expected result

Constant eastward ocean velocity (2.5 m/s), constant southward (-1.0 m/s), `K=0`. The
backward step must move the particle **west and north** (opposite the physical
velocity), by an amount matching `iste.geometry`'s own metre-to-degree conversion
exactly.

In [10]:
u_speed, v_speed = 2.5, -1.0
ocean_det = make_ocean(np.full((3, 3), u_speed), np.full((3, 3), v_speed))
before_det = WeightedParticleState.initialize(lat0=[20.0], lon0=[110.0])
after_det = backward_euler_maruyama_step(before_det, T0, DT, ocean_det, LAT, LON, ZERO_K, np.random.default_rng(1))
show("deterministic backward step", before_det, after_det)

expected_dx_m, expected_dy_m = -u_speed * DT, -v_speed * DT
expected_lon = 110.0 + expected_dx_m / metres_per_degree_longitude(20.0)
expected_lat = 20.0 + expected_dy_m / metres_per_degree_latitude()
print(f"analytically expected: lat={expected_lat:.8f}, lon={expected_lon:.8f}")
assert abs(after_det.lat[0] - expected_lat) < 1e-9 and abs(after_det.lon[0] - expected_lon) < 1e-9

deterministic backward step:
  before: lat=20.00000000, lon=110.00000000, log_w=0.000000, valid=True
  after:  lat=20.00809389, lon=109.97846664, log_w=0.000000, valid=True
analytically expected: lat=20.00809389, lon=109.97846664


## 2. Stochastic displacement (diffusion-only: `U=0`, constant `K`)

With `U=0` and constant `K` (so `grad(K)=0`), the displacement is exactly
`sqrt(2*K*dt) * xi` for the RNG's own draw -- reproduced independently below.

In [11]:
ocean_zero = make_ocean(ZERO_UV, ZERO_UV)
k_const = np.full((3, 3), 50.0)
before_stoch = WeightedParticleState.initialize(lat0=[20.0], lon0=[110.0])
after_stoch = backward_euler_maruyama_step(before_stoch, T0, DT, ocean_zero, LAT, LON, k_const, np.random.default_rng(123))
show("stochastic (diffusion-only) step, seed=123", before_stoch, after_stoch)

xi = np.random.default_rng(123).standard_normal(size=(1, 2))
scale = np.sqrt(2.0 * 50.0 * DT)
expected_lon2 = 110.0 + (scale * xi[0, 0]) / metres_per_degree_longitude(20.0)
expected_lat2 = 20.0 + (scale * xi[0, 1]) / metres_per_degree_latitude()
print(f"independently reconstructed from the same seed: lat={expected_lat2:.8f}, lon={expected_lon2:.8f}")
assert abs(after_stoch.lat[0] - expected_lat2) < 1e-9
assert after_stoch.log_weight[0] == 0.0  # div(U)=0 for a constant field

stochastic (diffusion-only) step, seed=123:
  before: lat=20.00000000, lon=110.00000000, log_w=0.000000, valid=True
  after:  lat=19.99900772, lon=109.99716012, log_w=0.000000, valid=True
independently reconstructed from the same seed: lat=19.99900772, lon=109.99716012


## 3. Reproducibility: same seed vs. a different seed

In [12]:
same_a = backward_euler_maruyama_step(
    WeightedParticleState.initialize(lat0=[20.0], lon0=[110.0]), T0, DT, ocean_zero, LAT, LON, k_const, np.random.default_rng(7)
)
same_b = backward_euler_maruyama_step(
    WeightedParticleState.initialize(lat0=[20.0], lon0=[110.0]), T0, DT, ocean_zero, LAT, LON, k_const, np.random.default_rng(7)
)
different = backward_euler_maruyama_step(
    WeightedParticleState.initialize(lat0=[20.0], lon0=[110.0]), T0, DT, ocean_zero, LAT, LON, k_const, np.random.default_rng(8)
)
print(f"seed=7 run A: lon={same_a.lon[0]:.10f}")
print(f"seed=7 run B: lon={same_b.lon[0]:.10f}  (identical to run A: {same_a.lon[0] == same_b.lon[0]})")
print(f"seed=8:       lon={different.lon[0]:.10f}  (different: {same_a.lon[0] != different.lon[0]})")

seed=7 run A: lon=110.0000035319
seed=7 run B: lon=110.0000035319  (identical to run A: True)
seed=8:       lon=109.9950092387  (different: True)


## 4. Log-weight update: zero, convergent, and divergent flow

In [13]:
# zero divergence (spatially uniform U)
ocean_uniform = make_ocean(np.full((3, 3), 0.4), np.full((3, 3), -0.2))
out_zero_div = backward_euler_maruyama_step(
    WeightedParticleState.initialize(lat0=[20.0], lon0=[110.0], log_weight0=1.5), T0, DT, ocean_uniform, LAT, LON, ZERO_K, np.random.default_rng(4)
)
print(f"zero-divergence flow: log_w before=1.5, after={out_zero_div.log_weight[0]:.6f} (expect unchanged)")

# convergent flow: uo = -C*lon
C = 0.001
uo_conv = -C * LON[None, :] * np.ones((3, 1))
out_conv = backward_euler_maruyama_step(
    WeightedParticleState.initialize(lat0=[20.0], lon0=[110.0]), T0, DT, make_ocean(uo_conv, ZERO_UV), LAT, LON, ZERO_K, np.random.default_rng(6)
)
print(f"convergent flow (div(U)<0): log_w after={out_conv.log_weight[0]:.6f} (expect > 0, weight increases)")

# divergent flow: uo = +C*lon
uo_div = C * LON[None, :] * np.ones((3, 1))
out_div = backward_euler_maruyama_step(
    WeightedParticleState.initialize(lat0=[20.0], lon0=[110.0]), T0, DT, make_ocean(uo_div, ZERO_UV), LAT, LON, ZERO_K, np.random.default_rng(6)
)
print(f"divergent flow (div(U)>0):  log_w after={out_div.log_weight[0]:.6f} (expect < 0, weight decreases)")
assert out_conv.log_weight[0] > 0 > out_div.log_weight[0]

zero-divergence flow: log_w before=1.5, after=1.500000 (expect unchanged)
convergent flow (div(U)<0): log_w after=0.000009 (expect > 0, weight increases)
divergent flow (div(U)>0):  log_w after=-0.000009 (expect < 0, weight decreases)


## 5. grad(K) drift: moves up the diffusivity gradient

`K` linear in longitude, `K=0` exactly at the query point (isolates the drift from the
diffusion term, which is also exactly zero there since `K=0`).

In [14]:
C2 = 100.0
k_gradient = C2 * (LON[None, :] - 110.0) * np.ones((3, 1))
before_gradk = WeightedParticleState.initialize(lat0=[20.0], lon0=[110.0])
after_gradk = backward_euler_maruyama_step(before_gradk, T0, DT, ocean_zero, LAT, LON, k_gradient, np.random.default_rng(3))
show("grad(K) drift (U=0)", before_gradk, after_gradk)

dK_dx = C2 / metres_per_degree_longitude(20.0)
expected_lon_gradk = 110.0 + (dK_dx * DT) / metres_per_degree_longitude(20.0)
print(f"analytically expected lon: {expected_lon_gradk:.8f}")
assert after_gradk.lon[0] > 110.0
assert abs(after_gradk.lon[0] - expected_lon_gradk) < 1e-9

grad(K) drift (U=0):
  before: lat=20.00000000, lon=110.00000000, log_w=0.000000, valid=True
  after:  lat=20.00000000, lon=110.00000824, log_w=0.000000, valid=True
analytically expected lon: 110.00000824


## 6. Invalid-destination case: particle stopped at land

In [15]:
uo_push = np.full((3, 3), 50.0)
uo_push[0, 0] = np.nan  # land at (lat=10, lon=100)
ocean_land = make_ocean(uo_push, ZERO_UV)
before_land = WeightedParticleState.initialize(lat0=[10.0], lon0=[110.0], log_weight0=0.7)
after_land = backward_euler_maruyama_step(before_land, T0, DT * 20, ocean_land, LAT, LON, ZERO_K, np.random.default_rng(9))
show("strong backward push straight into a masked land cell", before_land, after_land)
assert not after_land.valid[0] and after_land.lat[0] == 10.0 and after_land.lon[0] == 110.0

strong backward push straight into a masked land cell:
  before: lat=10.00000000, lon=110.00000000, log_w=0.700000, valid=True
  after:  lat=10.00000000, lon=110.00000000, log_w=0.700000, valid=False


## 7. Already-invalid particle: attempted again, does not move

In [16]:
already_invalid = WeightedParticleState(lat=np.array([15.0]), lon=np.array([105.0]), valid=np.array([False]), log_weight=np.array([2.5]))
after_already_invalid = backward_euler_maruyama_step(already_invalid, T0, DT, make_ocean(np.full((3,3),5.0), np.full((3,3),5.0)), LAT, LON, np.full((3,3),30.0), np.random.default_rng(10))
show("already-invalid particle, attempted again", already_invalid, after_already_invalid)
assert not after_already_invalid.valid[0] and after_already_invalid.lat[0] == 15.0

already-invalid particle, attempted again:
  before: lat=15.00000000, lon=105.00000000, log_w=2.500000, valid=False
  after:  lat=15.00000000, lon=105.00000000, log_w=2.500000, valid=False


## Summary / known limitations

- The deterministic step, the stochastic (diffusion-only) step, the `grad(K)` drift, and
  the `div(U)` weight update all match their analytically expected values exactly (to
  floating-point precision).
- The backward-time sign convention is confirmed: a particle moves **opposite** the
  physical ocean velocity (`-U`, never `+U`).
- The same RNG seed reproduces an identical step; a different seed does not.
- A particle proposing a step into land (or leaving the domain, not separately shown here
  -- see T04's own notebook) is marked invalid, does not move, and its log-weight is
  frozen at its pre-step value; an already-invalid particle never moves again.
- `K` still has no defined physical value/formula -- every `K` field used above is a
  synthetic test field, exactly as the task requires.
- This is **one step only**: no multi-step trajectory (T06), no `observation_pixel_id`
  tagging (T07), no forcing ensemble (T08), no multi-particle/multi-member run (T09), no
  diagnostics module (T10), no storage (T11), no validation suite (T12), no sign-off
  artifact (T13), and no KDE/likelihood/SMC/MCMC/AIS exist in this repository.
- `div(U)` is computed from the two hourly `uo`/`vo` snapshots bracketing `t`, blended by
  the same fraction T02 uses for `U` itself; `K` (and therefore `grad(K)`) is treated as a
  single time-instant grid with no time-interpolation, since `K` is currently
  synthetic/test-only with no time-varying source -- an explicit, documented
  simplification, not a silent one.